# Right-censored data: targeted estimation (HAL-TMLE)

HAL-TMLE is the method we propose for univariate right-censored data. It starts from a HAL density fit of the
event time and updates it along a one-dimensional path so that the estimate of a chosen feature solves that
feature's efficient score equation. The targeted estimate comes with an influence-curve standard error. The
features available: survival at time points, the mean, the restricted mean survival time (RMST), the
squared-density functional (density L2) and the entropy.

In [1]:
import numpy as np
import pandas as pd
from scipy.integrate import quad
from scipy.stats import beta

from haldensity.datasets import make_right_censored
from haldensity.rc import KaplanMeier, fit_right_censored
from haldensity.rc.targeting import (
    right_censored_density_l2_targeting_M_step, right_censored_entropy_targeting_M_step,
    right_censored_mean_estimand_variance, right_censored_mean_targeting_M_step,
    right_censored_rmst_estimand_variance, right_censored_rmst_targeting_M_step,
    right_censored_survival_estimand_variance, right_censored_survival_targeting_M_step,
)

data = make_right_censored(400, seed=2)          # event times Beta(2.2, 2.5); censoring Unif[0, 1.2]
truth = beta(2.2, 2.5)
km = KaplanMeier().fit(data, time_col="T", delta_col="Delta")

`KaplanMeier` estimates the censoring survival P(C > t); the targeting steps use it for their weights.

## The initial fit

The default point estimator (`"cvl1mle"`) serves as the initial fit; `details["estimators"]` holds the live
estimator that targeting starts from. The summary has the columns of the first right-censored tutorial: the
fit's survival and density at each `t0`, its L1 bound (`norm_constraint`) and that bound divided by M1
(`norm_factor`).

In [2]:
points = np.array([0.2, 0.4, 0.6])
initial = fit_right_censored(data, points, method="cvl1mle", n_trials=20, random_state=0)
(initial_estimator,) = initial.details["estimators"].values()
initial.summary

,t0,survival,density,norm_factor,norm_constraint
0,0.2,0.869419,1.069681,1.1,75.532771
1,0.4,0.598388,1.444667,1.1,75.532771
2,0.6,0.334874,1.335090,1.1,75.532771


## Survival at chosen time points

In the targeted fit's summary `psi_init` is the initial fit's survival and `psi_star` the targeted one (the
TMLE). The Kaplan–Meier estimate of the event-time survival, computed below in a few lines, is shown for
comparison.

In [3]:
# Kaplan-Meier estimate of the event-time survival at the points `at` (no tied times)
def kaplan_meier(times, events, at):
    order = np.argsort(times)
    times, events = times[order], events[order]
    curve = np.cumprod(1.0 - events / (len(times) - np.arange(len(times))))
    return np.array([curve[times <= t][-1] if np.any(times <= t) else 1.0 for t in at])


survival = right_censored_survival_targeting_M_step(
    initial_estimator=initial_estimator, observed_data=data, targeting_points=points, km=km)
se = np.sqrt(right_censored_survival_estimand_variance(targeted_fit=survival, observed_data=data))
summary = survival["summary"]
pd.DataFrame({"t0": points, "initial": summary["psi_init"], "TMLE": summary["psi_star"], "SE": se,
              "95% lower": summary["psi_star"] - 1.96 * se, "95% upper": summary["psi_star"] + 1.96 * se,
              "Kaplan-Meier": kaplan_meier(data["T"].to_numpy(), data["Delta"].to_numpy(), points),
              "truth": truth.sf(points)}).round(4)

,t0,initial,TMLE,SE,95% lower,95% upper,Kaplan-Meier,truth
0,0.2,0.8694,0.8635,0.0184,0.8274,0.8996,0.8635,0.8887
1,0.4,0.5984,0.6060,0.0282,0.5506,0.6614,0.6060,0.6039
2,0.6,0.3349,0.3487,0.0304,0.2892,0.4083,0.3488,0.2853


Here the TMLE and the Kaplan–Meier estimate agree to about 1e-4 at each point.

## The mean and the restricted mean survival time

In [4]:
tau = 0.8
mean = right_censored_mean_targeting_M_step(initial_estimator=initial_estimator, observed_data=data, km=km)
rmst = right_censored_rmst_targeting_M_step(initial_estimator=initial_estimator, observed_data=data,
                                            targeting_points=[tau], km=km)
rows = {}
for name, fit, variance, truth_value in (
    ("mean", mean, right_censored_mean_estimand_variance, truth.mean()),
    (f"RMST({tau})", rmst, right_censored_rmst_estimand_variance, quad(truth.sf, 0, tau)[0]),
):
    estimate = fit["summary"]["psi_star"].iloc[0]
    se = float(np.sqrt(variance(targeted_fit=fit, observed_data=data))[0])
    rows[name] = {"initial": fit["summary"]["psi_init"].iloc[0], "TMLE": estimate, "SE": se,
                  "95% lower": estimate - 1.96 * se, "95% upper": estimate + 1.96 * se, "truth": truth_value}
pd.DataFrame(rows).T.round(4)

,initial,TMLE,SE,95% lower,95% upper,truth
mean,0.4759,0.4756,0.0134,0.4492,0.5019,0.4681
RMST(0.8),0.4705,0.4702,0.0127,0.4453,0.4951,0.4643


## Density L2 and entropy

Their standard errors are not shown: in a simulation of this design they were too small (the 95% intervals
covered the truth in 87–94% of samples for density L2 and 72–88% for the entropy), and the reason is being studied.

In [5]:
fits = {
    "density L2": right_censored_density_l2_targeting_M_step(initial_estimator=initial_estimator,
                                                             observed_data=data, km=km),
    "entropy": right_censored_entropy_targeting_M_step(initial_estimator=initial_estimator, observed_data=data, km=km),
}
true_values = {"density L2": quad(lambda t: truth.pdf(t) ** 2, 0, 1)[0], "entropy": truth.entropy()}
pd.DataFrame({name: {"initial": fit["summary"]["psi_init"].iloc[0], "TMLE": fit["summary"]["psi_star"].iloc[0],
                     "truth": true_values[name]} for name, fit in fits.items()}).T.round(4)

,initial,TMLE,truth
density L2,1.1852,1.2022,1.2882
entropy,-0.1123,-0.1218,-0.1816
